# CEB 2025 compatibility investigation


These are the recorded **pre-implementation** findings. Pipeline fixes now change the results of rerunning this notebook. See [the final integration review](../data-quality/ceb-2025-review.md) for current validation; the pre-refresh source baseline is archived in `audits/ceb-2025/before-refresh/clean/`.
Executed 9 October 2026 in the UN Transparency Portal repository. See [the integration plan](ceb-2025-integration-plan.md) for findings, interpretation and implementation sequence.

This notebook compares an immutable CEB candidate snapshot against the **promoted clean baseline**, profiles coverage/categories and executes actual pipeline modules inside a temporary directory. It writes only ignored audit evidence. The full-source revenue failure and obsolete expense-year assertion are expected findings; the 2025-only and append-to-history revenue runs pass.

Run with the repository's `uv` Python environment. Start in the repository root or this notebook's directory. The audited snapshot must exist at `audits/ceb-2025/snapshot/`. It was captured with `uv run python python/02-fetch_ceb_data.py --fetch-snapshot`, then moved from the emitted timestamped directory into that ignored audit location. **Do not overwrite promoted raw/clean sources to execute these checks.** A newly downloaded snapshot is a new investigation, not necessarily a reproduction of the recorded result.

Official sources: [downloads](https://www.unsceb.org/data-download/), [release notes](https://www.unsceb.org/statistics/financial-statistics/). All amounts are USD. Source repeated rows and signed corrections are retained; empty amounts are not certified as zero. Small control differences and unresolved mappings are discussed in the plan.

Recorded SHA-256 values for the nine inspected pipeline files:

- `expenses_by_country_region_sub_agency.csv`: `c074f7fb27eec99aeaf651d34a3aa092d5033e0e25ca5720709ec1786b828cf2`
- `expenses_sdgs.csv`: `4e1f7274bd65282bfd4b1264afa0566a0cad27a5f08a86642a0f7ec66c6221e0`
- `expenses_sub_agency.csv`: `75a71ac2fc005f12001c8f654b7ecc08f308880acded6e1883071ab698636eb3`
- `revenue.csv`: `fbb5954e5145c135a496c5cea468a0c66a8053e9944c69bdbdb499bf6d1f7210`
- `revenue_by_financing_instruments_and_government_contributors.csv`: `fe1393942131ea36b93ca594db2006782cd0ad49295abf587845b0b6df70de70`
- `revenue_contrib_type.csv`: `42bc290d438c8d1f9b11e424d843df9f17393f1f6d165738651fccd841973c58`
- `revenue_government_donors.csv`: `b2ffc351b922186098d2a8cc9caa729ee46464d70d9c3c1487078fccb8bd245f`
- `revenue_non_gov_donors.csv`: `1f5a76044e12f223a3187c01b1ea9e6904e5fe86dc598ee8b1c1985b519e38f0`
- `revenue_sub_agency.csv`: `3231d3a3f4854514aebbc71fca291e3e9757baf3815ca2af60ca2741ee5df883`

In [1]:
from pathlib import Path
import contextlib, hashlib, importlib.util, io, json, os, shutil, sys, tempfile
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / 'python/02-fetch_ceb_data.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Run inside the portal repository')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'python'))
from utils import normalize_entity, clean_donor_name
SNAPSHOT = ROOT / 'audits/ceb-2025/snapshot'
assert SNAPSHOT.exists(), 'Capture a snapshot with step 02 --fetch-snapshot and archive it here first'

def read(path):
    try:
        frame = pd.read_csv(path)
    except UnicodeDecodeError:
        frame = pd.read_csv(path, encoding='cp1252')
    frame.columns = [c.strip().lower().replace(' ', '_').replace('usd_amount', 'amount') for c in frame.columns]
    frame['amount'] = pd.to_numeric(frame.amount, errors='raise')
    return frame

def module(name, filename):
    spec = importlib.util.spec_from_file_location(name, ROOT / 'python' / filename)
    result = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(result)
    return result

fresh = {p.name: read(p) for p in sorted(SNAPSHOT.glob('*.csv')) if (ROOT / 'data/ceb/clean' / p.name).exists()}
BASELINE = ROOT / 'audits/ceb-2025/before-refresh/clean'
assert BASELINE.exists(), 'The original baseline archive is needed to compare historical revisions'
baseline = {name: read(BASELINE / name) for name in fresh}
summary = {'retrievedAt': '2026-10-09', 'files': {}}
for name, frame in fresh.items():
    old = baseline[name]
    ec = next(c for c in ('agency', 'entity', 'entity_code') if c in frame)
    selected = frame[frame.calendar_year == 2025]
    categories = {}
    for c in ('rev_type', 'sub_type', 'contrib_type', 'transaction_type', 'location_type', 'expenses_type', 'sdg_goal'):
        if c in frame:
            a = set(frame.loc[frame.calendar_year == 2024, c].dropna().astype(str))
            b = set(selected[c].dropna().astype(str))
            categories[c] = {'2024': sorted(a), '2025': sorted(b), 'added': sorted(b-a), 'removed': sorted(a-b)}
    # Compare the source release to the promoted CLEAN baseline, not unreconciled raw.
    def entity_controls(d):
        d = d.copy()
        d[ec] = d[ec].map(normalize_entity)
        return d.groupby(['calendar_year', ec]).amount.sum()
    a = entity_controls(frame[frame.calendar_year.isin(old.calendar_year.unique())])
    b = entity_controls(old)
    diff = a.subtract(b, fill_value=0)
    diff = diff[diff.abs() > 0.01]
    summary['files'][name] = {
        'sha256': hashlib.sha256((SNAPSHOT / name).read_bytes()).hexdigest(),
        'columns': list(frame), 'addedColumns': sorted(set(frame)-set(old)), 'removedColumns': sorted(set(old)-set(frame)),
        'years': sorted(int(y) for y in frame.calendar_year.unique()),
        'rows2025': len(selected), 'entities2025': int(selected[ec].nunique()),
        'total2025': float(selected.amount.sum()), 'negativeRows2025': int((selected.amount < 0).sum()),
        'nullAmountsAllYears': int(frame.amount.isna().sum()),
        'duplicateFullRows2025': int(selected.duplicated().sum()),
        'categories': categories,
        'historicalEntityChanges': [{'year': int(y), 'entity': e, 'difference': float(v)} for (y,e),v in diff.items()],
    }
geo = fresh['expenses_by_country_region_sub_agency.csv'].query('calendar_year == 2025')
totals = fresh['expenses_sub_agency.csv'].query('calendar_year == 2025').groupby('agency').amount.sum()
e06 = geo.query("expenses_type == 'E06'").groupby('agency').amount.sum()
parts = geo.query("expenses_type != 'E06'").groupby('agency').amount.sum()
summary['geography'] = {
    'byExpenseType': {k: {'rows': len(v), 'amount': float(v.amount.sum())} for k,v in geo.groupby('expenses_type')},
    'E06VsExpenseControls': {e:float(v) for e,v in (e06-totals).items() if abs(v) > 0.01},
    'E06VsComponentSum': {e:float(v) for e,v in (e06-parts).items() if abs(v) > 0.01},
}
ng = fresh['revenue_non_gov_donors.csv']
summary['nonGovernmentDonors'] = {
    str(y): sorted(set(ng.loc[ng.calendar_year == y, 'donor'].map(clean_donor_name))) for y in (2024,2025)
}
# Verify the promoted geography differs by location even where annual totals agree.
old = baseline['expenses_by_country_region_sub_agency.csv']
def geo_controls(frame):
    frame = frame.copy().fillna({'country/territory':'', 'region':''})
    frame['agency'] = frame.agency.map(normalize_entity)
    return frame.groupby(['calendar_year','agency','country/territory','location_type','region'],dropna=False).amount.sum()
diff = geo_controls(fresh['expenses_by_country_region_sub_agency.csv'].query('calendar_year >= 2013 and calendar_year <= 2024')).subtract(geo_controls(old),fill_value=0)
summary['geography']['historicalChangedLocationKeys'] = int((diff.abs() > 0.01).sum())
summary['geography']['largestHistoricalLocationChanges'] = [{'key':str(k),'difference':float(diff[k])} for k in diff.abs().nlargest(5).index]

# Exercise the actual modules in a temporary working directory; no public or promoted source files are written.
with tempfile.TemporaryDirectory(prefix='ceb-2025-dry-run-') as td:
    work = Path(td)
    for folder in ('data/ceb/clean','data/ceb/fused','public/data'):
        (work / folder).mkdir(parents=True)
    for name, frame in fresh.items():
        frame.to_csv(work / 'data/ceb/clean' / name,index=False)
    for name in ('contrib_types_mapping.csv','member_states.csv'):
        shutil.copy(ROOT / 'data/ceb' / name,work / 'data/ceb' / name)
    shutil.copy(ROOT / 'public/data/entities.json',work / 'public/data/entities.json')
    os.chdir(work)
    log = io.StringIO()
    try:
        with contextlib.redirect_stdout(log):
            revenue = module('ceb_revenue','03-fuse_ceb_revenue.py')
            try:
                revenue.fuse_revenue()
                historical_revenue_error = None
            except AssertionError as exc:
                historical_revenue_error = str(exc)
            # Test the 2025 release independently of historical revisions.
            for name, frame in fresh.items():
                frame[frame.calendar_year == 2025].to_csv(work / 'data/ceb/clean' / name,index=False)
            revenue.fuse_revenue()
            # Recommended release policy: retain promoted historical rows, append 2025.
            for name, frame in fresh.items():
                pd.concat([baseline[name], frame[frame.calendar_year == 2025]], ignore_index=True).to_csv(work / 'data/ceb/clean' / name,index=False)
            r = revenue.fuse_revenue()
            exp = module('ceb_expenses','06-fuse_ceb_expenses.py')
            try:
                exp.fuse_expenses()
                expense_error = None
            except AssertionError as exc:
                expense_error = str(exc)
            exports = module('ceb_donors','05-export_contributor_json.py')
            # Explore the smallest proposed fix without changing the production module.
            exports.YEARS = sorted(int(y) for y in r.year.unique())
            data, states = exports.load_data()
            exports.export_donors_json(data,states)
            exports.export_entity_revenue_json(data)
            exports.export_contributor_trends_json(data)
            expense_exports = module('ceb_expense_exports','07-export_expenses_json.py')
            expense_exports.export_entity_spending(expense_exports.load_fused_expenses())
            functions = module('ceb_functions','18-export_function_expenses.py').build_export(fresh['expenses_sub_agency.csv'])
            flows = module('ceb_flows','19-export_system_flows.py').build_export(
                fresh['expenses_sub_agency.csv'],fresh['expenses_by_country_region_sub_agency.csv'],fresh['expenses_sdgs.csv'])
        selected = r[r.year == 2025]
        unattributed = selected[selected.donor_type == 'Unattributed']
        controls = r[r.year == 2024].groupby('entity').amount.sum()
        shipped = json.loads((ROOT / 'public/data/entity-revenue-2024.json').read_text())
        changed = {e: float(v-shipped.get(e,{}).get('total',0)) for e,v in controls.items() if abs(v-shipped.get(e,{}).get('total',0))>0.01}
        summary['dryRun'] = {
            'revenueFusion': '2025-only and promoted history + 2025 both passed entity/instrument reconciliation',
            'expenseFusionError': expense_error,
            'exported2025EntityRevenue': json.loads((work / 'public/data/entity-revenue-2025.json').read_text()),
            '2025RevenueTotal': float(selected.amount.sum()),
            'unattributed2025Net': float(unattributed.amount.sum()),
            'unattributed2025Absolute': float(unattributed.amount.abs().sum()),
            'fullSnapshotRevenueError': historical_revenue_error,
            '2024EntityRevenueChangesVsShipped': changed,
            'functionYears': functions['meta']['years'], 'systemFlowYears': flows['meta']['years'],
            'files': sorted(p.name for p in (work/'public/data').glob('*2025.json')),
        }
    finally:
        os.chdir(ROOT)
( ROOT / 'audits/ceb-2025/dry-run.log').write_text(log.getvalue())
(ROOT / 'audits/ceb-2025/profile.json').write_text(json.dumps(summary,indent=2,allow_nan=False))
print(json.dumps({
    'coverage': {name: {'latestYear': max(item['years']), 'rows2025': item['rows2025'], 'addedColumns': item['addedColumns']} for name,item in summary['files'].items()},
    'geography2025': summary['geography']['byExpenseType'],
    'dryRun': {k:v for k,v in summary['dryRun'].items() if k != 'exported2025EntityRevenue'},
}, indent=2))


{
  "coverage": {
    "expenses_by_country_region_sub_agency.csv": {
      "latestYear": 2025,
      "rows2025": 30152,
      "addedColumns": [
        "expenses_type"
      ]
    },
    "expenses_sdgs.csv": {
      "latestYear": 2024,
      "rows2025": 0,
      "addedColumns": []
    },
    "expenses_sub_agency.csv": {
      "latestYear": 2025,
      "rows2025": 79,
      "addedColumns": []
    },
    "revenue.csv": {
      "latestYear": 2025,
      "rows2025": 16562,
      "addedColumns": []
    },
    "revenue_by_financing_instruments_and_government_contributors.csv": {
      "latestYear": 2025,
      "rows2025": 10411,
      "addedColumns": []
    },
    "revenue_contrib_type.csv": {
      "latestYear": 2025,
      "rows2025": 15464,
      "addedColumns": []
    },
    "revenue_government_donors.csv": {
      "latestYear": 2025,
      "rows2025": 8993,
      "addedColumns": []
    },
    "revenue_non_gov_donors.csv": {
      "latestYear": 2025,
      "rows2025": 968,
      "addedCo